In [3]:
# %% [1] Imports & HF login
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json, math, random, time, zipfile, re, pickle, gc, traceback, functools
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_recall_fscore_support

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device: cpu


In [4]:

# %% [2] Config
TECHNIQUES = ["full", "adapter", "lorma"]    # any of: full lorma lora adapter partial normtune kd (put "full" before "kd")
SEED = 42
SKIP_DONE = True
FINAL_ALL = False             # True = train on train+dev (no dev score) with the fixed epochs in HP -> final submissions

ARTIFACT_PATH = "/kaggle/input/datasets/akashmalgav2/arch1final"                        # tokenizer (legal_bpe.json)
MODEL_CKPT_PATH = "/kaggle/input/datasets/akashmalgav2/phase2-model1/model_best.pt"        # Arch #1 Phase-1 checkpoint

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
FILES = {
    "CL_train": "Phase_3_RR/Train/CL_train.json", "CL_dev": "Phase_3_RR/Train/CL_dev.json",
    "IT_train": "Phase_3_RR/Train/IT_train.json", "IT_dev": "Phase_3_RR/Train/IT_dev.json",
    "CL_test": "Phase_3_RR/Test/CL_test.json",    "IT_test": "Phase_3_RR/Test/IT_test.json",
}
VOCAB_FILE = "Phase_3_RR/Train/label_vocab.json"

MAX_SENT_TOKENS = 128         # longer sentences are cut to their first 128 tokens
TRAIN_TARGET_TOKS = [640, 400]  # two window layouts per document (acts as augmentation)
INFER_TARGET_TOK = 640
CTX_TOK = 320                 # tokens of earlier sentences given as context to every window
USE_POSFEAT = True            # sentence-position embeddings
USE_DOMAIN = True             # CL / IT embedding
CLASS_WEIGHT = "sqrt"         # "none" | "sqrt" | "inv"

BATCH_SIZE = 8
EVAL_BATCH = 8
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
DROPOUT_FT = 0.1

LORMA_R, LORMA_ALPHA = 8, 8
ADAPTER_DIM = 32
PARTIAL_LAYERS = 4
STUDENT_KEEP = [0, 1, 3, 4, 6, 7, 9, 10, 11]   # KD student: 9 of 12 Phase-1 layers (a guess; the 6-layer student was too damaged in Phase 2)
KD_T, KD_ALPHA = 2.0, 0.5

HP = {   # starting points, not tuned
    "full":     dict(epochs=8,  lr_body=5e-5, lr_head=5e-4),
    "lorma":    dict(epochs=15, lr_body=4e-4, lr_head=1e-3),
    "lora":     dict(epochs=15, lr_body=4e-4, lr_head=1e-3),
    "adapter":  dict(epochs=15, lr_body=4e-4, lr_head=1e-3),
    "partial":  dict(epochs=10, lr_body=1e-4, lr_head=5e-4),
    "normtune": dict(epochs=20, lr_body=1e-3, lr_head=1e-3),
    "kd":       dict(epochs=10, lr_body=1e-4, lr_head=5e-4),
}
for _t in TECHNIQUES:
    assert _t in HP, _t
if "kd" in TECHNIQUES and "full" in TECHNIQUES:
    assert TECHNIQUES.index("full") < TECHNIQUES.index("kd"), "put 'full' before 'kd'"

ROOT_OUT = "/kaggle/working"
def out_dir_for(tech):
    return f"{ROOT_OUT}/cs781_phase3_arch1_{tech}_s{SEED}"
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)
print("Will run:", TECHNIQUES, "| seed:", SEED, "| final-all:", FINAL_ALL)

Will run: ['full', 'adapter', 'lorma'] | seed: 42 | final-all: False


In [5]:
# %% [3] Tokenizer (Phase-1 BPE) with a cached sentence encoder
with open(f"{ARTIFACT_PATH}/legal_bpe.json", "r", encoding="utf-8") as f:
    _tok = json.load(f)
vocab = _tok["itos"]
merges = [tuple(p) for p in _tok["merges"]]
token_to_id = {tok: i for i, tok in enumerate(vocab)}
merges_rank = {p: i for i, p in enumerate(merges)}
PAD_ID, UNK_ID = token_to_id["<pad>"], token_to_id["<unk>"]
BOS_ID, EOS_ID = token_to_id["<s>"], token_to_id["</s>"]
SEP_ID = EOS_ID                       # </s> closes every sentence; its hidden state represents the sentence
VOCAB_SIZE = len(vocab)
EOW = "\u0100"

def word_to_symbols(word):
    return tuple(chr(b) for b in word.encode("utf-8", "ignore")) + (EOW,)

def bpe_encode_word(word):
    symbols = list(word_to_symbols(word))
    while len(symbols) > 1:
        pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
        ranked = [(merges_rank[p], p) for p in pairs if p in merges_rank]
        if not ranked:
            break
        _, best = min(ranked)
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == best[0] and symbols[i + 1] == best[1]:
                new_symbols.append(best[0] + best[1]); i += 2
            else:
                new_symbols.append(symbols[i]); i += 1
        symbols = new_symbols
    return symbols

@functools.lru_cache(maxsize=None)
def encode_word_cached(word):
    return tuple(token_to_id.get(sym, UNK_ID) for sym in bpe_encode_word(word))

def encode_sentence(text):
    ids = []
    for w in str(text).split():
        ids.extend(encode_word_cached(w))
        if len(ids) >= MAX_SENT_TOKENS:
            break
    return ids[:MAX_SENT_TOKENS]

In [7]:

# %% [4] Load Phase-3 data

def hf_local(path):
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=path,
        repo_type="dataset",
        token=HF_TOKEN
    )


def parse_vocab(raw):
    if isinstance(raw, list):
        labs = list(raw)

    elif isinstance(raw, dict) and raw:
        vals = list(raw.values())

        if all(isinstance(v, int) and not isinstance(v, bool) for v in vals):
            # label -> id
            labs = [
                k for k, _ in sorted(
                    raw.items(), key=lambda kv: kv[1]
                )
            ]
        else:
            # id -> label
            try:
                labs = [
                    v for _, v in sorted(
                        raw.items(), key=lambda kv: int(kv[0])
                    )
                ]
            except ValueError:
                labs = vals

    else:
        raise ValueError("Unsupported label_vocab.json format")

    labs = ["None" if l is None else str(l) for l in labs]

    assert len(labs) == 13 and len(set(labs)) == 13, labs

    return labs


LABELS = parse_vocab(
    json.load(open(hf_local(VOCAB_FILE), encoding="utf-8"))
)
LAB2ID = {l: i for i, l in enumerate(LABELS)}
NUM_CLS = len(LABELS)

print("Labels:", LABELS)

N_REL = 20
ABS_EDGES = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256, 512])
N_ABS = len(ABS_EDGES) + 1


def load_docs(path, domain):
    raw = json.load(open(hf_local(path), encoding="utf-8"))

    items = []

    if isinstance(raw, list):
        for i, d in enumerate(raw):
            if not isinstance(d, dict):
                raise ValueError(
                    f"{path}: record at index {i} is {type(d)}, "
                    "expected a dictionary"
                )
            items.append((str(d.get("id", i)), d))

    elif isinstance(raw, dict):
        for k, v in raw.items():

            # Some test files store the document's sentences
            # directly as a list under its document ID.
            if isinstance(v, list):
                d = {
                    "id": str(k),
                    "text": v
                }

            elif isinstance(v, dict):
                d = dict(v)
                d.setdefault("id", str(k))

            else:
                raise ValueError(
                    f"{path}: unexpected record type {type(v)} "
                    f"for document ID {k}"
                )

            items.append((str(d["id"]), d))

    else:
        raise ValueError(
            f"{path}: unexpected top-level type {type(raw)}"
        )

    docs = []

    for did, d in items:
        if "text" not in d:
            raise ValueError(
                f"{path}: document {did} has no 'text' field. "
                f"Available keys: {list(d.keys())}"
            )

        sents = d["text"]

        if isinstance(sents, str):
            raise ValueError(
                f"{path}: 'text' is a string, expected a list "
                f"of sentences (document {did})"
            )

        if not isinstance(sents, list):
            raise ValueError(
                f"{path}: 'text' must be a list for document {did}; "
                f"got {type(sents)}"
            )

        labs = d.get("labels")

        if labs is not None:
            assert len(labs) == len(sents), (
                f"{path} doc {did}: {len(labs)} labels vs "
                f"{len(sents)} sentences"
            )

            labs = [
                LAB2ID["None" if l is None else str(l)]
                for l in labs
            ]

        sent_ids = [encode_sentence(s) + [SEP_ID] for s in sents]
        n = len(sent_ids)

        if n == 0:
            continue

        prel = [
            min(int(N_REL * i / n), N_REL - 1)
            for i in range(n)
        ]

        pabs = np.searchsorted(
            ABS_EDGES, np.arange(n), side="right"
        ).tolist()

        docs.append({
            "id": did,
            "domain": domain,
            "sent_ids": sent_ids,
            "lens": [len(s) for s in sent_ids],
            "labels": labs,
            "n": n,
            "prel": prel,
            "pabs": pabs
        })

    return docs


t0 = time.time()

train_recs = (
    load_docs(FILES["CL_train"], 0)
    + load_docs(FILES["IT_train"], 1)
)

dev_recs = (
    load_docs(FILES["CL_dev"], 0)
    + load_docs(FILES["IT_dev"], 1)
)

test_recs = (
    load_docs(FILES["CL_test"], 0)
    + load_docs(FILES["IT_test"], 1)
)

print(
    f"Loaded in {time.time() - t0:.0f}s | "
    f"docs: train {len(train_recs)} "
    f"dev {len(dev_recs)} test {len(test_recs)}"
)

for name, rs in [
    ("train", train_recs),
    ("dev", dev_recs),
    ("test", test_recs)
]:
    for dom, dn in [(0, "CL"), (1, "IT")]:
        sub = [r for r in rs if r["domain"] == dom]

        if sub:
            ns = [r["n"] for r in sub]

            print(
                f"  {name} {dn}: {len(sub)} docs | "
                f"sentences {sum(ns)} | "
                f"per doc median {int(np.median(ns))} "
                f"max {max(ns)}"
            )

tok_per_sent = np.mean([
    l for r in train_recs for l in r["lens"]
])

print(
    f"Average tokens per sentence (incl. </s>): "
    f"{tok_per_sent:.1f}"
)

train_counts = Counter(
    l for r in train_recs for l in r["labels"]
)

print(
    "Train label counts:",
    {
        LABELS[i]: train_counts.get(i, 0)
        for i in range(NUM_CLS)
    }
)

all_ids = [r["id"] for r in test_recs]

assert len(all_ids) == len(set(all_ids)), (
    "test doc IDs collide across CL / IT"
)

Labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
Loaded in 1s | docs: train 80 dev 10 test 10
  train CL: 40 docs | sentences 10593 | per doc median 120 max 1816
  train IT: 40 docs | sentences 6252 | per doc median 126 max 442
  dev CL: 5 docs | sentences 1396 | per doc median 184 max 848
  dev IT: 5 docs | sentences 746 | per doc median 120 max 382
  test CL: 5 docs | sentences 1339 | per doc median 114 max 942
  test IT: 5 docs | sentences 858 | per doc median 159 max 420
Average tokens per sentence (incl. </s>): 37.5
Train label counts: {'Fact': 4098, 'Issue': 107, 'ArgumentPetitioner': 1613, 'ArgumentRespondent': 1348, 'Statute': 1099, 'Dissent': 284, 'PrecedentReliedUpon': 1945, 'PrecedentNotReliedUpon': 569, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 874, 'RatioOfTheDecision': 4301, 'Ruli

In [9]:
# %% [X1] Load all saved probability files
import pickle, json, zipfile, os, itertools
import numpy as np
from collections import Counter
from sklearn.metrics import f1_score
K = NUM_CLS

DEV_PKLS = {   # EDIT: runs trained on TRAIN only (have dev + test probs)
    "full":     "/kaggle/input/datasets/akashmalgav2/dev-pkls/sentence_probs_full.pkl",
    "lorma":    "/kaggle/input/datasets/akashmalgav2/dev-pkls/sentence_probs_lorma.pkl",
    "adapter":  "/kaggle/input/datasets/akashmalgav2/dev-pkls/sentence_probs_adapter.pkl",
    "kd":       "/kaggle/input/datasets/akashmalgav2/dev-pkls/sentence_probs_kd.pkl",
    "sentonly": "/kaggle/input/datasets/akashmalgav2/dev-pkls/sentence_probs_ctx_ablation.pkl",
}
ALL_PKLS = {   # EDIT: runs trained on train+dev (test probs only); same weight as the same-named run above
    "full":    "/kaggle/input/datasets/akashmalgav2/all-pkls/sentence_probs_full_all.pkl",
    "lorma":   "/kaggle/input/datasets/akashmalgav2/all-pkls/sentence_probs_lorma_all.pkl",
    "adapter": "/kaggle/input/datasets/akashmalgav2/all-pkls/sentence_probs_adapter_all.pkl",
}
runs = {}
for k, p in DEV_PKLS.items():
    d = pickle.load(open(p, "rb")); assert d["labels"] == LABELS, k; assert d["dev"], k; runs[k] = d
allruns = {}
for k, p in ALL_PKLS.items():
    if os.path.exists(p):
        d = pickle.load(open(p, "rb")); assert d["labels"] == LABELS, k; allruns[k] = d
    else: print("skip", p)
dev_ids  = [r["id"] for r in dev_recs]; test_ids = [r["id"] for r in test_recs]
gold = {r["id"]: np.array(r["labels"]) for r in dev_recs}

def build_trans(recs):
    trans = np.full((K, K), 0.5); start = np.full(K, 0.5); prior = np.full(K, 0.5)
    for r in recs:
        l = r["labels"]; start[l[0]] += 1
        for a in l: prior[a] += 1
        for a, b in zip(l[:-1], l[1:]): trans[a, b] += 1
    pp = prior / prior.sum()
    return (np.log(trans / trans.sum(1, keepdims=True)) - np.log(pp)[None, :],
            np.log(start / start.sum()) - np.log(pp))

def decode(P, lam, tr):
    pmi, spmi = tr
    lp = np.log(np.clip(P, 1e-8, 1.0)); n = len(lp)
    if lam == 0 or n == 1: return lp.argmax(1)
    score = lp[0] + lam * spmi; back = np.zeros((n, K), dtype=np.int64)
    for t in range(1, n):
        cand = score[:, None] + lam * pmi
        back[t] = cand.argmax(0); score = cand.max(0) + lp[t]
    path = np.zeros(n, dtype=np.int64); path[-1] = score.argmax()
    for t in range(n - 1, 0, -1): path[t - 1] = back[t, path[t]]
    return path

def dev_score(P, lam, tr):
    yt = np.concatenate([gold[i] for i in dev_ids])
    yp = np.concatenate([decode(P[i], lam, tr) for i in dev_ids])
    return f1_score(yt, yp, labels=list(range(K)), average="macro", zero_division=0)

def mix(split, ids, w):
    tot = sum(w.values())
    return {i: sum(w[k] * runs[k][split][i].astype(np.float32) for k in w if w[k] > 0) / tot for i in ids}



In [10]:
# %% [X2] Weight search on dev (argmax first) -- prints the TOP 8, not just the best, because dev is noisy
tr_train = build_trans(train_recs)
names = list(runs); res = []
for ws in itertools.product([0, 1, 2], repeat=len(names)):
    if sum(ws) == 0: continue
    w = dict(zip(names, ws)); res.append((dev_score(mix("dev", dev_ids, w), 0, tr_train), w))
res.sort(key=lambda x: -x[0])
for s, w in res[:8]: print(f"{s:.4f}", w)
print("equal weights:", round(dev_score(mix("dev", dev_ids, {k: 1 for k in names}), 0, tr_train), 4))

# %% [X3] Pick weights (default = best row) and lambda
W_FINAL = dict(res[0][1])          # or paste your own, e.g. {"full":2,"lorma":1,"adapter":2,"kd":1,"sentonly":1}
P_dev = mix("dev", dev_ids, W_FINAL)
best = (-1, 0)
for lam in [0, 0.25, 0.5, 0.75, 1.0]:
    s = dev_score(P_dev, lam, tr_train); print("lam", lam, round(s, 4)); best = max(best, (s, lam))
LAM = best[1]; print("weights", W_FINAL, "lambda", LAM)

# %% [X4] Final test prediction: train-only runs + (same weight) train+dev runs
members = []
for k, w in W_FINAL.items():
    if w > 0:
        members.append((runs[k]["test"], w))
        if k in allruns: members.append((allruns[k]["test"], w))
tot = sum(w for _, w in members); tr_all = build_trans(train_recs + dev_recs)
pred = {}
for r in test_recs:
    P = sum(w * d[r["id"]].astype(np.float32) for d, w in members) / tot
    labs = [LABELS[int(k)] for k in decode(P, LAM, tr_all)]
    assert len(labs) == r["n"]; pred[r["id"]] = labs
assert set(pred) == set(test_ids) and all(l in LAB2ID for v in pred.values() for l in v)
print("predicted label counts:", dict(Counter(l for v in pred.values() for l in v).most_common()))
OUT = "/kaggle/working/phase3_ens_v2"; os.makedirs(OUT, exist_ok=True)
json.dump(pred, open(f"{OUT}/prediction.json", "w"))
with zipfile.ZipFile(f"{OUT}/submission_phase3_ens_v2.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(f"{OUT}/prediction.json", arcname="prediction.json")
print("wrote", f"{OUT}/submission_phase3_ens_v2.zip")

0.4782 {'full': 0, 'lorma': 1, 'adapter': 2, 'kd': 1, 'sentonly': 1}
0.4755 {'full': 1, 'lorma': 1, 'adapter': 2, 'kd': 1, 'sentonly': 1}
0.4751 {'full': 1, 'lorma': 1, 'adapter': 2, 'kd': 2, 'sentonly': 1}
0.4726 {'full': 0, 'lorma': 1, 'adapter': 2, 'kd': 0, 'sentonly': 1}
0.4722 {'full': 1, 'lorma': 0, 'adapter': 2, 'kd': 1, 'sentonly': 0}
0.4710 {'full': 0, 'lorma': 1, 'adapter': 2, 'kd': 2, 'sentonly': 1}
0.4709 {'full': 1, 'lorma': 0, 'adapter': 2, 'kd': 1, 'sentonly': 1}
0.4692 {'full': 0, 'lorma': 1, 'adapter': 2, 'kd': 2, 'sentonly': 2}
equal weights: 0.4649
lam 0 0.4782
lam 0.25 0.4815
lam 0.5 0.4791
lam 0.75 0.479
lam 1.0 0.4703
weights {'full': 0, 'lorma': 1, 'adapter': 2, 'kd': 1, 'sentonly': 1} lambda 0.25
predicted label counts: {'RatioOfTheDecision': 631, 'Fact': 384, 'PrecedentReliedUpon': 379, 'ArgumentPetitioner': 294, 'Statute': 181, 'RulingByLowerCourt': 126, 'ArgumentRespondent': 67, 'RulingByPresentCourt': 45, 'PrecedentNotReliedUpon': 37, 'None': 23, 'Dissent': 